# 09 — Signaling Pathway Databases

This notebook demonstrates how to load cell-cell signaling interaction databases
(CellPhoneDB, CellChatDB) and convert them into pathway gene sets for subtype discovery.

**What you'll learn:**
- Load CellPhoneDB interactions (auto-download from GitHub)
- Load CellChatDB interactions (user-exported CSV)
- Convert interactions to pathway gene sets
- Merge multiple signaling databases
- Use signaling pathways with expression scoring and clustering

> **RESEARCH USE ONLY** — Not for clinical decision-making.

In [ ]:
# Install (skip if already installed)
# !pip install pathway-subtyping==0.3.1

In [ ]:
import numpy as np
import pandas as pd

from pathway_subtyping import (
    SignalingDatabase,
    SignalingDatabaseResult,
    SignalingInteraction,
    convert_interactions_to_pathways,
    load_cellchatdb,
    load_cellphonedb,
    merge_signaling_databases,
)

print("Imports OK")

## 1. Understanding the Data Format

Signaling databases contain ligand-receptor interaction records. The framework
groups these by signaling pathway classification and converts them into the
standard `Dict[str, List[str]]` format used throughout the pipeline.

```
Interactions → Group by pathway → Union of genes → Dict[pathway → [gene1, gene2, ...]]
```

## 2. Load CellPhoneDB (Auto-Download)

CellPhoneDB provides CSV data on GitHub. The loader automatically downloads
three files:
- `interaction_input.csv` — ligand-receptor pairs with classification
- `gene_input.csv` — UniProt-to-gene-symbol mapping
- `complex_input.csv` — multi-subunit complex definitions

For this demo, we'll create mock data to avoid network calls.

In [ ]:
from pathlib import Path
import tempfile

# Create mock CellPhoneDB data for demonstration
tmpdir = Path(tempfile.mkdtemp())

# Mock interaction_input.csv
interactions_csv = tmpdir / "interaction_input.csv"
interactions_csv.write_text(
    "id_cp_interaction,partner_a,partner_b,protein_name_a,protein_name_b,"
    "annotation_strategy,source,is_ppi,classification\n"
    "CPI-001,P12830,P22223,CADH1,CADH2,curated,literature,True,Adhesion by Cadherin\n"
    "CPI-002,P12830,integrin_complex,CADH1,Integrin,curated,literature,False,Adhesion by Cadherin\n"
    "CPI-003,P04628,Q13224,WNT1,FZD1,curated,literature,True,WNT signaling\n"
    "CPI-004,P56704,Q13224,WNT3A,FZD1,curated,literature,True,WNT signaling\n"
    "CPI-005,P00533,P21860,EGFR,ERBB3,curated,literature,True,EGF signaling\n"
    "CPI-006,P01133,P00533,EGF,EGFR,curated,literature,True,EGF signaling\n"
    "CPI-007,O00548,P46531,DLL1,NOTCH1,curated,literature,True,Notch signaling\n"
    "CPI-008,P78504,O00548,JAG1,DLL1,curated,literature,True,Notch signaling\n"
    "CPI-009,Q9Y6N7,Q9Y6N7,ROBO1,SLIT2,curated,literature,True,Axon guidance\n"
    "CPI-010,P36888,P36888,FLT3,FLT3LG,curated,literature,True,Growth factor\n"
)

# Mock gene_input.csv
gene_csv = tmpdir / "gene_input.csv"
gene_csv.write_text(
    "gene_name,uniprot,hgnc_symbol,ensembl\n"
    "CDH1,P12830,CDH1,ENSG00000039068\n"
    "CDH2,P22223,CDH2,ENSG00000170558\n"
    "WNT1,P04628,WNT1,ENSG00000125084\n"
    "WNT3A,P56704,WNT3A,ENSG00000154342\n"
    "FZD1,Q13224,FZD1,ENSG00000157404\n"
    "EGFR,P00533,EGFR,ENSG00000146648\n"
    "ERBB3,P21860,ERBB3,ENSG00000065361\n"
    "EGF,P01133,EGF,ENSG00000138798\n"
    "DLL1,O00548,DLL1,ENSG00000198719\n"
    "NOTCH1,P46531,NOTCH1,ENSG00000148400\n"
    "JAG1,P78504,JAG1,ENSG00000101384\n"
    "ROBO1,Q9Y6N7,ROBO1,ENSG00000169855\n"
    "SLIT2,Q9Y6N7,SLIT2,ENSG00000145423\n"
    "FLT3,P36888,FLT3,ENSG00000122025\n"
    "FLT3LG,P36888,FLT3LG,ENSG00000090554\n"
    "ITGA1,P11111,ITGA1,ENSG00000213949\n"
    "ITGB1,P22222,ITGB1,ENSG00000150093\n"
)

# Mock complex_input.csv
complex_csv = tmpdir / "complex_input.csv"
complex_csv.write_text(
    "complex_name,uniprot_1,uniprot_2,uniprot_3,uniprot_4,uniprot_5\n"
    "integrin_complex,P11111,P22222,,,\n"
)

print(f"Mock files created in {tmpdir}")

In [ ]:
# Load CellPhoneDB from local files
cpdb_result = load_cellphonedb(
    interactions_path=interactions_csv,
    gene_path=gene_csv,
    complex_path=complex_csv,
    min_genes_per_pathway=2,
)

print(f"Database: {cpdb_result.database.value}")
print(f"Interactions: {cpdb_result.n_interactions}")
print(f"Pathway gene sets: {cpdb_result.n_pathways}")
print(f"Unique genes: {cpdb_result.n_unique_genes}")
print()

# Show all pathway gene sets
for name, genes in sorted(cpdb_result.pathway_gene_sets.items()):
    print(f"  {name}: {genes}")

In [ ]:
# Display the formatted report
print(cpdb_result.format_report())

In [ ]:
# Check citations
for citation in cpdb_result.get_citations():
    print(citation)

## 3. Load CellChatDB (User-Exported CSV)

CellChatDB distributes data as R binary `.rda` files. Export from R first:

```r
library(CellChat)
write.csv(CellChatDB.human$interaction, "cellchatdb_human.csv", row.names = FALSE)
```

For this demo, we create a mock CellChatDB CSV.

In [ ]:
# Create mock CellChatDB CSV
cellchat_csv = tmpdir / "cellchatdb_human.csv"
cellchat_csv.write_text(
    "interaction_name,pathway_name,ligand,receptor,annotation,evidence\n"
    "COL1A1_CD44,COLLAGEN,COL1A1,CD44,Secreted Signaling,KEGG\n"
    "COL1A2_CD44,COLLAGEN,COL1A2,CD44,Secreted Signaling,KEGG\n"
    "COL3A1_DDR1,COLLAGEN,COL3A1,DDR1,Secreted Signaling,KEGG\n"
    "WNT5A_FZD4,WNT,WNT5A,FZD4,Secreted Signaling,KEGG\n"
    "WNT5B_FZD6,WNT,WNT5B,FZD6,Secreted Signaling,KEGG\n"
    "DLL1_NOTCH1,NOTCH,DLL1,NOTCH1,Cell-Cell Contact,KEGG\n"
    "JAG1_NOTCH1,NOTCH,JAG1,NOTCH1,Cell-Cell Contact,KEGG\n"
    "JAG2_NOTCH2,NOTCH,JAG2,NOTCH2,Cell-Cell Contact,KEGG\n"
    "BMP2_BMPR1A,BMP,BMP2,BMPR1A,Secreted Signaling,KEGG\n"
    "BMP4_BMPR2,BMP,BMP4,BMPR2,Secreted Signaling,KEGG\n"
)

# Load CellChatDB
ccdb_result = load_cellchatdb(cellchat_csv)

print(f"Database: {ccdb_result.database.value}")
print(f"Interactions: {ccdb_result.n_interactions}")
print(f"Pathway gene sets: {ccdb_result.n_pathways}")
print(f"Unique genes: {ccdb_result.n_unique_genes}")
print()

for name, genes in sorted(ccdb_result.pathway_gene_sets.items()):
    print(f"  {name}: {genes}")

## 4. Merge Multiple Databases

When both databases define the same pathway (e.g., "Signaling: WNT"),
their gene sets are unioned for broader coverage.

In [ ]:
# Merge CellPhoneDB + CellChatDB
merged = merge_signaling_databases(cpdb_result, ccdb_result)

print(f"Total merged pathway gene sets: {len(merged)}")
print()
for name, genes in sorted(merged.items()):
    print(f"  {name} ({len(genes)} genes): {genes}")

## 5. Custom Grouping

By default, `load_cellphonedb()` groups by the `classification` column.
You can also group by `annotation` for finer granularity using
`convert_interactions_to_pathways()`.

In [ ]:
# Re-group CellChatDB by annotation instead of pathway_name
by_annotation = convert_interactions_to_pathways(
    ccdb_result.interactions,
    group_by="annotation",
    min_genes=2,
    prefix="CellChat",
)

print(f"Grouped by annotation: {len(by_annotation)} groups")
for name, genes in sorted(by_annotation.items()):
    print(f"  {name} ({len(genes)} genes): {genes}")

## 6. Use with Expression Scoring

Signaling pathway gene sets are `Dict[str, List[str]]` — the same format
used by `score_pathways_from_expression()`, `run_clustering()`, and
all validation gates. Here's how to integrate them into a full workflow.

In [ ]:
from pathway_subtyping import run_clustering

# Create synthetic expression data with genes from our signaling pathways
all_genes = sorted(set(g for genes in merged.values() for g in genes))
n_samples = 60
np.random.seed(42)

# Simulate 3 subtypes with different signaling profiles
expr_data = np.random.randn(n_samples, len(all_genes))
# Subtype 0: high WNT
wnt_idx = [all_genes.index(g) for g in merged.get("Signaling: WNT signaling", merged.get("Signaling: WNT", [])) if g in all_genes]
expr_data[:20, wnt_idx] += 2.0
# Subtype 1: high NOTCH
notch_genes = [g for g in merged.get("Signaling: Notch signaling", merged.get("Signaling: NOTCH", [])) if g in all_genes]
notch_idx = [all_genes.index(g) for g in notch_genes]
expr_data[20:40, notch_idx] += 2.0

expr_df = pd.DataFrame(expr_data, columns=all_genes,
                        index=[f"S{i:03d}" for i in range(n_samples)])

# Compute pathway scores manually (mean-Z approach for simplicity)
pathway_scores = pd.DataFrame(index=expr_df.index)
for pw_name, pw_genes in merged.items():
    present = [g for g in pw_genes if g in expr_df.columns]
    if len(present) >= 2:
        scores = expr_df[present].mean(axis=1)
        pathway_scores[pw_name] = (scores - scores.mean()) / (scores.std() + 1e-10)

print(f"Pathway scores shape: {pathway_scores.shape}")
print(f"Pathways scored: {list(pathway_scores.columns)}")

# Cluster
clustering = run_clustering(pathway_scores.values, n_clusters=3, seed=42)
print(f"\nCluster sizes: {np.bincount(clustering.labels)}")

## 7. Inspect Individual Interactions

The `interactions` list in `SignalingDatabaseResult` gives access to
every parsed interaction for advanced analysis.

In [ ]:
# Show first 5 interactions from CellPhoneDB
for i, ix in enumerate(cpdb_result.interactions[:5]):
    print(f"{i+1}. {ix.interaction_id}: {ix.partner_a} ↔ {ix.partner_b}")
    print(f"   Pathway: {ix.pathway_name}")
    print(f"   Genes: {ix.all_genes}")
    print()

In [ ]:
# Serialize to dict (JSON-compatible)
import json

result_dict = cpdb_result.to_dict()
print(json.dumps(result_dict, indent=2))

## Summary

| Function | Source | Auto-Download | Output |
|----------|--------|---------------|--------|
| `load_cellphonedb()` | CellPhoneDB GitHub | Yes | `SignalingDatabaseResult` |
| `load_cellchatdb()` | User-exported CSV | No | `SignalingDatabaseResult` |
| `convert_interactions_to_pathways()` | Any interactions | — | `Dict[str, List[str]]` |
| `merge_signaling_databases()` | Multiple results | — | `Dict[str, List[str]]` |

All outputs are `Dict[str, List[str]]` — directly compatible with
`score_pathways_from_expression()`, `run_clustering()`, and validation gates.

**Next steps:**
- [02_expression_scoring.ipynb](02_expression_scoring.ipynb) — Score pathways from expression
- [03_multi_omic_fusion.ipynb](03_multi_omic_fusion.ipynb) — Combine with other modalities
- [API reference](../../docs/api/signaling_databases.md) — Full function docs